# 04 — Search Method Comparison

This notebook tests the three configurable search methods available in the Product Recommendation Agent:

| Method | Description | Requires Pinecone? |
|---|---|---|
| `bm25` | Keyword search (BM25 Okapi) | No |
| `hybrid` | BM25 + Pinecone vector search with RRF fusion | Falls back to BM25 if unavailable |
| `pinecone` | Pure semantic vector search | Yes |

We run the same set of queries through each method and compare results.

## Setup

In [ ]:
import sys
import time
import logging
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

# Enable logging to see index load times
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
    datefmt="%H:%M:%S",
)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from src.agents.product_agent import ProductRecommendationAgent
from src.agents.product_agent.config import VALID_SEARCH_METHODS, BM25_CACHE_PATH

print(f"Available search methods: {sorted(VALID_SEARCH_METHODS)}")
print(f"BM25 cache path: {BM25_CACHE_PATH}")
print(f"BM25 cache exists: {BM25_CACHE_PATH.exists()}")

## Test 1: BM25 Index Persistence

Verify that the BM25 index is saved to disk on first build and loaded from cache on subsequent runs.

> **Expected:** First run builds from DB/CSV (~30s). Second run loads from pickle (~1s).

In [ ]:
from src.search.bm25_index import BM25ProductIndex, get_bm25_index

# Time the singleton initialisation
t0 = time.perf_counter()
index = get_bm25_index()
elapsed = time.perf_counter() - t0

print(f"\nBM25 index loaded: {len(index)} products in {elapsed:.2f}s")
print(f"Cache file exists: {BM25_CACHE_PATH.exists()}")
if BM25_CACHE_PATH.exists():
    size_mb = BM25_CACHE_PATH.stat().st_size / (1024 * 1024)
    print(f"Cache file size: {size_mb:.1f} MB")

## Test 2: Direct BM25 Search (No Agent)

Test the BM25 index directly to verify keyword matching works.

In [ ]:
test_queries = [
    "gaming headset with surround sound",
    "wireless earbuds long battery life",
    "quiet washing machine",
    "laptop for students under 500",
    "4K television with HDR",
]

for query in test_queries:
    results = index.search(query, top_k=3)
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    if results:
        for i, r in enumerate(results, 1):
            title = str(r.get('title', ''))[:70]
            price = r.get('price', 'N/A')
            score = r.get('_bm25_score', 0)
            rating = r.get('average_rating', 'N/A')
            print(f"  {i}. [{score:.2f}] {title}")
            print(f"     Price: ${price} | Rating: {rating}")
    else:
        print("  (no results)")

## Test 3: Agent with `search_method='bm25'`

Create an agent using BM25-only search and test it with a natural language query.

In [ ]:
agent_bm25 = ProductRecommendationAgent(
    session_id="test-bm25",
    search_method="bm25",
    use_twitter_samples=False,
    debug=True,
)

print(f"Agent created with search_method='{agent_bm25.search_method}'")
print()

response = agent_bm25.chat("Show me the top 5 gaming headsets under $100")
print("\n--- Final Response ---")
print(response)

## Pinecone Status Check

Before testing hybrid and pinecone search methods, let's verify the Pinecone connection and index status.

This checks:
1. Whether `PINECONE_API_KEY` is configured in `.env`
2. Whether the Pinecone index exists and is reachable
3. How many vectors are currently indexed
4. Index dimensions and metric

In [ ]:
import os
from src.embeddings.vector_store import PineconeVectorStore

store = PineconeVectorStore()

# 1. Check API key
api_key = os.getenv("PINECONE_API_KEY", "")
index_name = os.getenv("PINECONE_INDEX_NAME", "product-catalog")
key_set = bool(api_key) and api_key != "your-pinecone-api-key-here"

print("=" * 60)
print("Pinecone Status Check")
print("=" * 60)
print(f"  API Key configured : {'YES' if key_set else 'NO (set PINECONE_API_KEY in .env)'}")
print(f"  Index name         : {index_name}")
print(f"  is_available()     : {store.is_available()}")

# 2. Try to get index stats
if store.is_available():
    print("\nConnecting to Pinecone...")
    stats = store.index_stats()
    if "error" in stats:
        print(f"  ERROR: {stats['error']}")
    else:
        total_vectors = getattr(stats, 'total_vector_count', None) or stats.get('total_vector_count', 'N/A')
        dimension = getattr(stats, 'dimension', None) or stats.get('dimension', 'N/A')
        print(f"  Connected!")
        print(f"  Total vectors      : {total_vectors}")
        print(f"  Dimension          : {dimension}")
        print(f"  Full stats         : {stats}")
        
        # 3. Quick search test
        print("\nRunning quick Pinecone search test...")
        try:
            from src.embeddings.embedder import ProductEmbedder
            embedder = ProductEmbedder()
            test_vec = embedder.embed_texts(["wireless headphones"])[0].tolist()
            pinecone_results = store.query(embedding=test_vec, top_k=3)
            if pinecone_results:
                print(f"  Search returned {len(pinecone_results)} results:")
                for i, r in enumerate(pinecone_results, 1):
                    title = str(r.get('title', ''))[:60]
                    score = r.get('_vector_score', 0)
                    print(f"    {i}. [score:{score:.4f}] {title}")
            else:
                print("  Search returned 0 results (index may be empty).")
        except Exception as e:
            print(f"  Search test failed: {e}")
else:
    print("\n  Pinecone is NOT available.")
    print("  -> Hybrid search will fall back to BM25-only.")
    print("  -> Pinecone-only search will return an error.")
    print("\n  To enable: set PINECONE_API_KEY in .env and run:")
    print("    python pipelines/03_index_product_catalog.py --max-rows 1000")

print("\n" + "=" * 60)

## Test 4: Agent with `search_method='hybrid'`

Create an agent using hybrid (BM25 + Pinecone) search.

> **Note:** If Pinecone is not configured, this will gracefully fall back to BM25-only.

In [ ]:
agent_hybrid = ProductRecommendationAgent(
    session_id="test-hybrid",
    search_method="hybrid",
    use_twitter_samples=False,
    debug=True,
)

print(f"Agent created with search_method='{agent_hybrid.search_method}'")
print()

response = agent_hybrid.chat("Show me the top 5 gaming headsets under $100")
print("\n--- Final Response ---")
print(response)

## Test 5: Agent with `search_method='pinecone'`

Create an agent using Pinecone-only semantic search.

> **Requires:** `PINECONE_API_KEY` set in `.env` and vectors indexed via `pipelines/03_index_product_catalog.py`.
> If Pinecone is not set up, the agent will return a clear error message.

In [ ]:
agent_pinecone = ProductRecommendationAgent(
    session_id="test-pinecone",
    search_method="pinecone",
    use_twitter_samples=False,
    debug=True,
)

print(f"Agent created with search_method='{agent_pinecone.search_method}'")
print()

response = agent_pinecone.chat("Show me the top 5 gaming headsets under $100")
print("\n--- Final Response ---")
print(response)

## Test 6: Semantic vs Keyword Query Comparison

This is the key test: queries where semantic understanding matters.

BM25 matches keywords literally, while vector search understands intent.

| Query | BM25 expectation | Vector expectation |
|---|---|---|
| "something to watch movies in bed" | Poor (no keyword match for 'tablet'/'projector') | Should find tablets, projectors |
| "quiet appliance for small kitchen" | Poor (matches 'quiet' literally) | Should find dishwashers, small appliances |

In [ ]:
semantic_queries = [
    "something to watch movies in bed",
    "quiet appliance for small kitchen",
    "gift for someone who likes photography",
]

print("Comparing BM25 vs Hybrid for semantic queries...")
print("(Hybrid falls back to BM25 if Pinecone is not configured)\n")

for query in semantic_queries:
    print(f"\n{'='*70}")
    print(f"Query: \"{query}\"")
    print(f"{'='*70}")
    
    # BM25 results
    bm25_results = index.search(query, top_k=3)
    print(f"\n  BM25 Results:")
    if bm25_results:
        for i, r in enumerate(bm25_results, 1):
            title = str(r.get('title', ''))[:60]
            score = r.get('_bm25_score', 0)
            print(f"    {i}. [{score:.2f}] {title}")
    else:
        print("    (no results)")
    
    # Hybrid results (uses whatever is available)
    from src.search.hybrid_search import hybrid_search
    hybrid_results = hybrid_search(query=query, top_k=3)
    print(f"\n  Hybrid Results:")
    if hybrid_results:
        for i, r in enumerate(hybrid_results, 1):
            title = str(r.get('title', ''))[:60]
            rrf = r.get('_rrf_score', 0)
            print(f"    {i}. [rrf:{rrf:.4f}] {title}")
    else:
        print("    (no results)")

## Test 7: Invalid Search Method (Error Handling)

Verify that passing an invalid search method raises a clear error.

In [ ]:
try:
    bad_agent = ProductRecommendationAgent(
        search_method="invalid_method",
    )
    print("ERROR: Should have raised ValueError!")
except ValueError as e:
    print(f"Correctly raised ValueError: {e}")

## Test 8: BM25 Cache Reload Test

Force-reload the BM25 index from the pickle cache and verify it's fast.

In [ ]:
if BM25_CACHE_PATH.exists():
    t0 = time.perf_counter()
    reloaded = BM25ProductIndex.load_from_disk(BM25_CACHE_PATH)
    elapsed = time.perf_counter() - t0
    
    print(f"Reloaded from cache: {len(reloaded)} products in {elapsed:.2f}s")
    
    # Quick sanity check 
    results = reloaded.search("wireless mouse", top_k=2)
    for r in results:
        print(f"  - {r.get('title', '')[:60]} (score: {r.get('_bm25_score', 0):.2f})")
else:
    print("No BM25 cache file found. Run a BM25 search first to create it.")

---

## Summary

| Test | What was tested |
|---|---|
| 1 | BM25 index persistence (save/load pickle) |
| 2 | Direct BM25 search (keyword matching) |
| 3 | Agent with `search_method='bm25'` |
| — | **Pinecone status check** (API key, connection, vector count, quick search) |
| 4 | Agent with `search_method='hybrid'` (BM25 + Pinecone RRF) |
| 5 | Agent with `search_method='pinecone'` (pure semantic) |
| 6 | Semantic vs keyword query comparison |
| 7 | Invalid search method error handling |
| 8 | BM25 cache reload speed test |

### Key Takeaways

- **BM25** is fast and works out of the box. Great for exact keyword queries but struggles with semantic intent.
- **Hybrid** gives the best results by combining keyword + semantic search. Falls back to BM25 if Pinecone is not set up.
- **Pinecone** is pure semantic search. Best for intent-based queries but requires the indexing pipeline to be run first.
- BM25 index persistence saves ~30s on subsequent kernel restarts.